In [0]:
%run ../00_common/data_utils

In [0]:
def calc_consumer_phone_master(task_id):
    """
    计算并更新consumer phone主数据
    支持所有Region的SQL逻辑,包括:
    1. 标准的DELETE操作处理
    2. TWN Region的source system code额外过滤
    
    Args:
        task_id: 任务ID
    """
    master_phone_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_phone"
    # 1. 初始化数据源
    itermediate_consumer_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer") \
        .where(f"TASK_ID = '{task_id}'") \
        .filter(F.col("IS_INCLUDE") == True) 

    itermediate_phone_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_phone") \
        .where(f"TASK_ID = '{task_id}'")

    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_phone_df = spark.table(master_phone_table_name)
    
    # TWN Region特殊表: tsourcesystemcodesurvivelatesttimestamp
    source_system_survive_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_sourcesystemcode_survive")
    
    # 2. 查询query_phone (步骤4提前)
    # 构建基础DataFrame - 包含所有必要的join
    base_phone_df = (
        itermediate_consumer_df.alias("srcc")
        .join(
            itermediate_phone_df.alias("srcp"),
            (F.col("srcc.srcc_id") == F.col("srcp.srcp_srcc_id")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srcp.srcp_mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srcp.srcp_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srcp.srcp_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_phone_df.alias("scph"),
            (F.col("scon.scon_id") == F.col("scph.scph_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scph.scph_mrkt_code")) &
            (F.col("srcp.srcp_phnt_code") == F.col("scph.scph_phnt_code")),
            "left"
        )
        .join(
            source_system_survive_df.alias("sss"),
            (F.col("srcc.srcc_srcs_code") == F.col("sss.srcs_code")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("sss.market_code")),
            "left"
        )
    )
    
    # 应用过滤条件: 标准条件 + TWN特殊条件
    master_phone_to_insert = (
        base_phone_df
        .where(
            # 标准条件: phone number, quality code 或 country code 有变化
            (
                (F.coalesce(F.col("srcp.srcp_phonenumber"), F.lit("")) != 
                 F.coalesce(F.col("scph.scph_phonenumber"), F.lit(""))) |
                (F.coalesce(F.col("srcp.srcp_quality_code"), F.lit("")) != 
                 F.coalesce(F.col("scph.scph_quality_code"), F.lit(""))) |
                (F.coalesce(F.col("srcp.SRCP_PHONECOUNTRYCODE"), F.lit("")) != 
                 F.coalesce(F.col("scph.scph_phonecountrycode"), F.lit("")))
            ) |
            # TWN特殊条件: source system code在whitelist中
            (
                (F.col("srcc.srcc_mrkt_code") == "TWN") &
                F.col("sss.srcs_code").isNotNull()
            )
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srcp.srcp_mrkt_code"),
            F.col("srcp.srcp_phnt_code"),
            # 动态timestamp: DELETE时使用srcc_sourcetimestamp
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.col("srcc.srcc_sourcetimestamp")
            ).otherwise(F.col("srcp.srcp_sourcetimestamp")).alias("srcp_sourcetimestamp"),
            F.col("srcp.SRCP_PHONECOUNTRYCODE").alias("srcp_phonecountrycode"),
            # 动态phonenumber: DELETE时清空
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.lit("")
            ).otherwise(F.col("srcp.srcp_phonenumber")).alias("srcp_phonenumber"),
            F.col("srcp.srcp_validitycode"),
            F.col("srcp.srcp_primary_flag"),
            F.col("srcp.SRCP_QUALITY_CODE"),
            F.col("srcp.SRCP_QUALITY_DESC"),
            F.col("srcp.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scph_id"),
            F.col("scon_id").alias("scph_scon_id"),
            F.col("srcp_mrkt_code").alias("scph_mrkt_code"),
            F.col("srcp_phnt_code").alias("scph_phnt_code"),
            F.col("srcp_sourcetimestamp").alias("scph_sourcetimestamp"),
            F.col("srcp_phonecountrycode").alias("scph_phonecountrycode"),
            F.col("srcp_phonenumber").alias("scph_phonenumber"),
            F.col("srcp_validitycode").alias("scph_validitycode"),
            F.col("srcp_primary_flag").alias("scph_primary_flag"),
            F.col("SRCP_QUALITY_CODE").alias("scph_quality_code"),
            F.col("SRCP_QUALITY_DESC").alias("scph_quality_desc"),
            F.current_timestamp().alias("scph_creation_dt"),
            F.lit("ELC").alias("scph_creation_uid"),
            F.current_timestamp().alias("scph_update_dt"),
            F.lit("ELC").alias("scph_update_uid"),
            F.lit(True).alias("scph_update_flag"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_phone_to_insert = master_phone_to_insert.checkpoint(eager=True)
    master_phone_insert_count = master_phone_to_insert.count()
    
    # 3. 查询query_phone_delete
    # 注意：只有TWN才需要join srcc表，其他Market直接从srcp join到scon
    master_phone_to_delete = (
        itermediate_phone_df.alias("srcp")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srcp.srcp_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srcp.srcp_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_phone_df.alias("scph"),
            (F.col("scon.scon_id") == F.col("scph.scph_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scph.scph_mrkt_code")) &
            (F.col("srcp.srcp_phnt_code") == F.col("scph.scph_phnt_code")),
            "inner"
        )
        .join(
            itermediate_consumer_df.alias("srcc"),
            (F.col("srcp.srcp_mrkt_code") == F.col("srcc.srcc_mrkt_code")) &  # 先join市场code，减少数据量
            (F.col("srcp.srcp_srcc_id") == F.col("srcc.srcc_id")),
            "left"  # left join: 只有TWN需要srcc，其他Market不受影响
        )
        .join(
            source_system_survive_df.alias("sss"),
            (F.col("srcc.srcc_srcs_code") == F.col("sss.srcs_code")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("sss.market_code")),
            "left"
        )
        .where(
            # 标准删除条件
            (
                (F.coalesce(F.col("srcp.srcp_phonenumber"), F.lit("")) != 
                 F.coalesce(F.col("scph.scph_phonenumber"), F.lit(""))) |
                (F.coalesce(F.col("srcp.srcp_quality_code"), F.lit("")) != 
                 F.coalesce(F.col("scph.scph_quality_code"), F.lit(""))) |
                (F.coalesce(F.col("srcp.SRCP_PHONECOUNTRYCODE"), F.lit("")) != 
                 F.coalesce(F.col("scph.scph_phonecountrycode"), F.lit("")))
            ) |
            # TWN特殊条件
            (
                (F.col("srcc.srcc_mrkt_code") == "TWN") &
                F.col("sss.srcs_code").isNotNull()
            )
        )
        .select(F.col("scph.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_phone_delete_count = 0
    if not master_phone_to_delete.isEmpty():
        master_phone_delta_table = DeltaTable.forName(spark, master_phone_table_name)
        master_phone_merge_result = (
            master_phone_delta_table.alias("target")
            .merge(
                master_phone_to_delete.alias("source"),
                """
                target.scph_id = source.scph_id AND 
                target.scph_mrkt_code = source.scph_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_phone_delete_count = max(master_phone_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_phone_insert_count > 0:
        append_table(master_phone_to_insert, master_phone_table_name)
    print(f"master phone inserted count: {master_phone_insert_count}, deleted count: {master_phone_delete_count}")

In [0]:
def calc_consumer_address_master(task_id):
    """
    计算并更新consumer address主数据
    支持所有Region的SQL逻辑,包括:
    1. 标准的DELETE操作处理
    2. TWN Region的source system code额外过滤
    
    Args:
        task_id: 任务ID
    """
    master_address_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_address"
    # 1. 初始化数据源
    itermediate_consumer_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer") \
        .where(f"TASK_ID = '{task_id}'") \
        .filter(F.col("IS_INCLUDE") == True) 

    itermediate_address_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_address") \
        .where(f"TASK_ID = '{task_id}'")

    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_address_df = spark.table(master_address_table_name)
    
    # TWN Region特殊表: tsourcesystemcodesurvivelatesttimestamp
    source_system_survive_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_sourcesystemcode_survive")
    
    # 2. 查询query_address (步骤4提前)
    # 构建基础DataFrame - 包含所有必要的join
    base_address_df = (
        itermediate_consumer_df.alias("srcc")
        .join(
            itermediate_address_df.alias("srca"),
            (F.col("srcc.srcc_id") == F.col("srca.srca_srcc_id")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srca.srca_mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srca.srca_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srca.srca_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_address_df.alias("scad"),
            (F.col("scon.scon_id") == F.col("scad.scad_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scad.scad_mrkt_code")) &
            (F.col("srca.srca_addt_code") == F.col("scad.scad_addt_code")),
            "left"
        )
        .join(
            source_system_survive_df.alias("sss"),
            (F.col("srcc.srcc_srcs_code") == F.col("sss.srcs_code")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("sss.market_code")),
            "left"
        )
    )
    
    # 应用过滤条件: 标准条件 + TWN特殊条件
    master_address_to_insert = (
        base_address_df
        .where(
            # 标准条件: address字段有变化
            (
                (F.coalesce(F.col("srca.srca_address1"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_address1"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_address2"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_address2"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_address3"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_address3"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_city_localdesc"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_city_localdesc"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_prvn_localdesc"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_prvn_localdesc"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_postalcode"), F.lit("0")) != 
                 F.coalesce(F.col("scad.scad_postalcode"), F.lit("0"))) |
                (F.coalesce(F.col("srca.srca_quality_code"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_quality_code"), F.lit("inv")))
            ) |
            # TWN特殊条件: source system code在whitelist中
            (
                (F.col("srcc.srcc_mrkt_code") == "TWN") &
                F.col("sss.srcs_code").isNotNull()
            )
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srca.srca_mrkt_code"),
            F.col("srca.srca_addt_code"),
            # 动态timestamp: DELETE时使用srcc_sourcetimestamp
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.col("srcc.srcc_sourcetimestamp")
            ).otherwise(F.col("srca.srca_sourcetimestamp")).alias("srca_sourcetimestamp"),
            # 动态address字段: DELETE时清空
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.lit("")
            ).otherwise(F.col("srca.srca_address1")).alias("srca_address1"),
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.lit("")
            ).otherwise(F.col("srca.srca_address2")).alias("srca_address2"),
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.lit("")
            ).otherwise(F.col("srca.srca_address3")).alias("srca_address3"),
            F.col("srca.srca_city_localdesc"),
            F.col("srca.srca_prvn_localdesc"),
            F.col("srca.srca_cntr_isoalpha3code"),
            F.col("srca.srca_postalcode"),
            F.col("srca.srca_validitycode"),
            F.col("srca.srca_primary_flag"),
            F.col("srca.srca_quality_code"),
            F.col("srca.srca_quality_desc"),
            F.col("srca.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scad_id"),
            F.col("scon_id").alias("scad_scon_id"),
            F.col("srca_mrkt_code").alias("scad_mrkt_code"),
            F.col("srca_addt_code").alias("scad_addt_code"),
            F.col("srca_sourcetimestamp").alias("scad_sourcetimestamp"),
            F.col("srca_address1").alias("scad_address1"),
            F.col("srca_address2").alias("scad_address2"),
            F.col("srca_address3").alias("scad_address3"),
            F.col("srca_city_localdesc").alias("scad_city_localdesc"),
            F.col("srca_prvn_localdesc").alias("scad_prvn_localdesc"),
            F.col("srca_cntr_isoalpha3code").alias("scad_cntr_isoalpha3code"),
            F.col("srca_postalcode").alias("scad_postalcode"),
            F.col("srca_validitycode").alias("scad_validitycode"),
            F.col("srca_primary_flag").alias("scad_primary_flag"),
            F.col("srca_quality_code").alias("scad_quality_code"),
            F.col("srca_quality_desc").alias("scad_quality_desc"),
            F.current_timestamp().alias("scad_creation_dt"),
            F.lit("ELC").alias("scad_creation_uid"),
            F.current_timestamp().alias("scad_update_dt"),
            F.lit("ELC").alias("scad_update_uid"),
            F.lit(True).alias("scad_update_flag"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_address_to_insert = master_address_to_insert.checkpoint(eager=True)
    master_address_insert_count = master_address_to_insert.count()
    
    # 3. 查询query_address_delete
    # 注意：只有TWN才需要join srcc表，其他Market直接从srca join到scon
    master_address_to_delete = (
        itermediate_address_df.alias("srca")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srca.srca_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srca.srca_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_address_df.alias("scad"),
            (F.col("scon.scon_id") == F.col("scad.scad_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scad.scad_mrkt_code")) &
            (F.col("srca.srca_addt_code") == F.col("scad.scad_addt_code")),
            "inner"
        )
        .join(
            itermediate_consumer_df.alias("srcc"),
            (F.col("srca.srca_mrkt_code") == F.col("srcc.srcc_mrkt_code")) &  # 先join市场code，减少数据量
            (F.col("srca.srca_srcc_id") == F.col("srcc.srcc_id")),
            "left"  # left join: 只有TWN需要srcc，其他Market不受影响
        )
        .join(
            source_system_survive_df.alias("sss"),
            (F.col("srcc.srcc_srcs_code") == F.col("sss.srcs_code")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("sss.market_code")),
            "left"
        )
        .where(
            # 标准删除条件
            (
                (F.coalesce(F.col("srca.srca_address1"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_address1"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_address2"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_address2"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_address3"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_address3"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_city_localdesc"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_city_localdesc"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_prvn_localdesc"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_prvn_localdesc"), F.lit(""))) |
                (F.coalesce(F.col("srca.srca_postalcode"), F.lit("0")) != 
                 F.coalesce(F.col("scad.scad_postalcode"), F.lit("0"))) |
                (F.coalesce(F.col("srca.srca_quality_code"), F.lit("")) != 
                 F.coalesce(F.col("scad.scad_quality_code"), F.lit("inv")))
            ) |
            # TWN特殊条件
            (
                (F.col("srcc.srcc_mrkt_code") == "TWN") &
                F.col("sss.srcs_code").isNotNull()
            )
        )
        .select(F.col("scad.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_address_delete_count = 0
    if not master_address_to_delete.isEmpty():
        master_address_delta_table = DeltaTable.forName(spark, master_address_table_name)
        master_address_merge_result = (
            master_address_delta_table.alias("target")
            .merge(
                master_address_to_delete.alias("source"),
                """
                target.scad_id = source.scad_id AND 
                target.scad_mrkt_code = source.scad_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_address_delete_count = max(master_address_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_address_insert_count > 0:
        append_table(master_address_to_insert, master_address_table_name)
    print(f"master address inserted count: {master_address_insert_count}, deleted count: {master_address_delete_count}")

In [0]:
def calc_consumer_optin_master(task_id):
    """
    计算并更新consumer optin主数据
    支持所有Region的SQL逻辑,包括:
    1. 标准的optin_flag变化检测
    2. HKG Region的source system code白名单过滤
    3. 与media/phone/address表的update_flag关联检测
    
    Args:
        task_id: 任务ID
    """
    master_optin_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_optin"
    
    # 1. 初始化数据源
    itermediate_consumer_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer") \
        .where(f"TASK_ID = '{task_id}'") \
        .filter(F.col("IS_INCLUDE") == True) 

    itermediate_optin_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_optin") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_consumer")
    master_optin_df = spark.table(master_optin_table_name)
    
    # HKG Region特殊表: sourcesystemcodesurvivelatesttimestamp (原表名 sourcesystemcodeoptintimestampchange)
    hkg_optin_timestamp_change_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_sourcesystemcode_survive")
    
    # media/phone/address表用于检查update_flag
    master_emedia_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_emedia")
    master_phone_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_phone")
    master_address_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_master_address")
    
    # 2. 查询query_optin (步骤4提前)
    # 构建基础DataFrame
    base_optin_df = (
        itermediate_consumer_df.alias("srcc")
        .join(
            itermediate_optin_df.alias("srco"),
            (F.col("srcc.srcc_id") == F.col("srco.srco_srcc_id")) &
            (F.col("srcc.srcc_mrkt_code") == F.col("srco.srco_mrkt_code")),
            "inner"
        )
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srco.srco_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srco.srco_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_optin_df.alias("scop"),
            (F.col("scon.scon_id") == F.col("scop.scop_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scop.scop_mrkt_code")) &
            (F.col("srco.srco_comm_code") == F.col("scop.scop_comm_code")),
            "left"
        )
        .join(
            master_emedia_df.alias("scme"),
            (F.col("scon.scon_id") == F.col("scme.scme_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scme.scme_mrkt_code")),
            "left"
        )
        .join(
            master_phone_df.alias("scph"),
            (F.col("scon.scon_id") == F.col("scph.scph_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scph.scph_mrkt_code")),
            "left"
        )
        .join(
            master_address_df.alias("scad"),
            (F.col("scon.scon_id") == F.col("scad.scad_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scad.scad_mrkt_code")),
            "left"
        )
        # HKG额外join白名单表
        .join(
            hkg_optin_timestamp_change_df.alias("hotc"),
            (F.col("srcc.srcc_srcs_code") == F.col("hotc.srcs_code")) & 
            (F.col("srcc.srcc_mrkt_code") == F.col("hotc.market_code")),
            "left"
        )
    )
    
    # 构建WHERE条件
    # 条件1: 基础optin_flag不同
    base_condition = (
        F.coalesce(F.col("srco.srco_optin_flag").cast("String"), F.lit("x")) != 
        F.coalesce(F.col("scop.scop_optin_flag").cast("String"), F.lit("x"))
    )
    
    # 条件2: HKG source system code在白名单中
    hkg_condition = F.col("hotc.srcs_code").isNotNull()
    
    # 条件3: eml/mbleml相关 + media update_flag
    media_condition = (
        F.col("srco.srco_comm_code").isin(["eml", "mbleml"]) &
        F.col("scme.scme_emdt_code").isin(["emlprs", "mblemlprs", "emlprf", "mblemlprf"]) &
        (F.col("scme.scme_update_flag") == True)
    )
    
    # 条件4: sms/mms相关 + phone update_flag
    phone_condition = (
        F.col("srco.srco_comm_code").isin(["sms", "mms"]) &
        F.col("scph.scph_phnt_code").isin(["mblprs", "mblprf"]) &
        (F.col("scph.scph_update_flag") == True)
    )
    
    # 条件5: drcml相关 + address update_flag
    address_condition = (
        F.col("srco.srco_comm_code").isin(["drcml"]) &
        (F.col("scad.scad_update_flag") == True)
    )
    
    # 组合所有条件
    combined_condition = (
        base_condition | 
        hkg_condition | 
        media_condition | 
        phone_condition | 
        address_condition
    )
    
    # 应用条件并选择字段
    master_optin_to_insert = (
        base_optin_df
        .where(combined_condition)
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srco.srco_mrkt_code"),
            # 动态optin_dt: DELETE时使用srcc_sourcetimestamp
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.col("srcc.srcc_sourcetimestamp")
            ).otherwise(F.col("srco.srco_optin_dt")).alias("srco_optin_dt"),
            F.col("srco.srco_comm_code"),
            # 动态optin_flag: DELETE时设为0
            F.when(
                F.upper(F.col("srcc.SRCC_ACTION")) == "DELETE",
                F.lit(False)
            ).otherwise(F.col("srco.srco_optin_flag")).alias("srco_optin_flag"),
            F.col("srco.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scop_id"),
            F.col("scon_id").alias("scop_scon_id"),
            F.col("srco_mrkt_code").alias("scop_mrkt_code"),
            F.col("srco_optin_dt").alias("scop_optin_dt"),
            F.col("srco_comm_code").alias("scop_comm_code"),
            F.col("srco_optin_flag").alias("scop_optin_flag"),
            F.current_timestamp().alias("scop_creation_dt"),
            F.lit("ELC").alias("scop_creation_uid"),
            F.current_timestamp().alias("scop_update_dt"),
            F.lit("ELC").alias("scop_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_optin_to_insert = master_optin_to_insert.checkpoint(eager=True)
    master_optin_insert_count = master_optin_to_insert.count()
    
    # 3. 查询query_optin_delete
    # 注意：只有HKG才需要join srcc表，其他Market直接从srco join到scon
    delete_base_df = (
        itermediate_optin_df.alias("srco")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srco.srco_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srco.srco_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .join(
            master_optin_df.alias("scop"),
            (F.col("scon.scon_id") == F.col("scop.scop_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scop.scop_mrkt_code")) &
            (F.col("srco.srco_comm_code") == F.col("scop.scop_comm_code")),
            "inner"
        )
        .join(
            master_emedia_df.alias("scme"),
            (F.col("scon.scon_id") == F.col("scme.scme_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scme.scme_mrkt_code")),
            "left"
        )
        .join(
            master_phone_df.alias("scph"),
            (F.col("scon.scon_id") == F.col("scph.scph_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scph.scph_mrkt_code")),
            "left"
        )
        .join(
            master_address_df.alias("scad"),
            (F.col("scon.scon_id") == F.col("scad.scad_scon_id")) &
            (F.col("scon.scon_mrkt_code") == F.col("scad.scad_mrkt_code")),
            "left"
        )
        # HKG额外join临时srcc主表
        .join(
            itermediate_consumer_df.alias("srcc"),
            (F.col("srco.srco_mrkt_code") == F.col("srcc.srcc_mrkt_code")) &
            (F.col("srco.srco_srcc_id") == F.col("srcc.srcc_id")),
            "left"
        )
        # HKG额外join白名单表
        .join(
            hkg_optin_timestamp_change_df.alias("hotc"),
            (F.col("srcc.srcc_srcs_code") == F.col("hotc.srcs_code")) & 
            (F.col("srcc.srcc_mrkt_code") == F.col("hotc.market_code")),
            "left"
        )
    )
    
    # WHERE条件与insert相同
    master_optin_to_delete = (
        delete_base_df
        .where(combined_condition)
        .select(F.col("scop.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_optin_delete_count = 0
    if not master_optin_to_delete.isEmpty():
        master_optin_delta_table = DeltaTable.forName(spark, master_optin_table_name)
        master_optin_merge_result = (
            master_optin_delta_table.alias("target")
            .merge(
                master_optin_to_delete.alias("source"),
                """
                target.scop_id = source.scop_id AND
                target.scop_mrkt_code = source.scop_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_optin_delete_count = max(master_optin_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_optin_insert_count > 0:
        append_table(master_optin_to_insert, master_optin_table_name)
    print(f"master optin inserted count: {master_optin_insert_count}, deleted count: {master_optin_delete_count}")

In [0]:
def calc_consumer_program_master(task_id):
    """
    计算并更新consumer program主数据
    所有Region逻辑一致：全量替换模式（删除特定scon_id的所有program记录后重新插入）
    
    Args:
        task_id: 任务ID
    """
    master_program_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_program"
    master_consumer_table_name = f"{get_env_config('golden_consumer_master_database')}.t_master_consumer"

    # 1. 初始化数据源
    itermediate_program_df = spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_program") \
        .where(f"TASK_ID = '{task_id}'")
        
    master_consumer_df = spark.table(master_consumer_table_name)
    master_program_df = spark.table(master_program_table_name)
    
    # 2. 查询query_program (步骤4提前)
    master_program_to_insert = (
        itermediate_program_df.alias("srpg")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("srpg.srpg_srcc_id") == F.col("scon.scon_srcc_id")) &
            (F.col("srpg.srpg_mrkt_code") == F.col("scon.scon_mrkt_code")),
            "inner"
        )
        .select(
            F.col("scon.consumermdmkey"),
            F.col("scon.scon_id"),
            F.col("srpg.srpg_mrkt_code"),
            F.col("srpg.SRPG_APPLICATION_TOCH_CODE"),
            F.col("srpg.SRPG_CONSUMER_GRP"),
            F.col("srpg.SRPG_PRGT_CODE"),
            F.col("srpg.SRPG_PRGT_DESC"),
            F.col("srpg.SRPG_PRGL_CODE"),
            F.col("srpg.SRPG_PRGL_DESC"),
            F.col("srpg.SRPG_SYSTEM_CODE"),
            F.col("srpg.SRPG_SYSTEM_DESC"),
            F.col("srpg.SRPG_MEMBERSHIPNUM"),
            F.col("srpg.SRPG_CARDNUM"),
            F.col("srpg.SRPG_START_DT"),
            F.col("srpg.SRPG_END_DT"),
            F.col("srpg.SRPG_ACQUIREDPOINT_NUM"),
            F.col("srpg.SRPG_REDEEMEDPOINT_NUM"),
            F.col("srpg.SRPG_INITIAL_QUOTA"),
            F.col("srpg.SRPG_AVAILABLE_QUOTA"),
            F.col("srpg.batch_id")
        )
        .distinct()
        .drop("consumermdmkey")
        .select(
            F.expr("uuid()").alias("scpr_id"),
            F.col("scon_id").alias("scpr_scon_id"),
            F.col("srpg_mrkt_code").alias("scpr_mrkt_code"),
            F.col("SRPG_APPLICATION_TOCH_CODE").alias("scpr_application_toch_code"),
            F.col("SRPG_CONSUMER_GRP").alias("scpr_consumer_grp"),
            F.col("SRPG_PRGT_CODE").alias("scpr_prgt_code"),
            F.col("SRPG_PRGT_DESC").alias("scpr_prgt_desc"),
            F.col("SRPG_PRGL_CODE").alias("scpr_prgl_code"),
            F.col("SRPG_PRGL_DESC").alias("scpr_prgl_desc"),
            F.col("SRPG_SYSTEM_CODE").alias("scpr_system_code"),
            F.col("SRPG_SYSTEM_DESC").alias("scpr_system_desc"),
            F.col("SRPG_MEMBERSHIPNUM").alias("scpr_membershipnum"),
            F.col("SRPG_CARDNUM").alias("scpr_cardnum"),
            F.col("SRPG_START_DT").alias("scpr_start_dt"),
            F.col("SRPG_END_DT").alias("scpr_end_dt"),
            F.col("SRPG_ACQUIREDPOINT_NUM").alias("scpr_acquiredpoint_num"),
            F.col("SRPG_REDEEMEDPOINT_NUM").alias("scpr_redeemedpoint_num"),
            F.col("SRPG_INITIAL_QUOTA").alias("scpr_initial_quota"),
            F.col("SRPG_AVAILABLE_QUOTA").alias("scpr_available_quota"),
            F.current_timestamp().alias("scpr_creation_dt"),
            F.lit("ELC").alias("scpr_creation_uid"),
            F.current_timestamp().alias("scpr_update_dt"),
            F.lit("ELC").alias("scpr_update_uid"),
            F.col("batch_id"),
            F.lit(task_id).alias("task_id")
        )
    )
    master_program_to_insert = master_program_to_insert.checkpoint(eager=True)
    master_program_insert_count = master_program_to_insert.count()
    
    # 3. 查询query_program_delete
    # 先获取需要删除的scon_id列表
    scon_ids_to_delete = (
        itermediate_program_df.alias("srpg")
        .join(
            master_consumer_df.alias("sc"),
            (F.col("sc.scon_mrkt_code") == F.col("srpg.SRPG_MRKT_CODE")) &
            (F.col("sc.scon_srcc_id") == F.col("srpg.SRPG_SRCC_ID")),
            "inner"
        )
        .select(F.col("sc.scon_id"), F.col("sc.scon_mrkt_code"))
        .distinct()
    )
    
    # 获取要删除的完整program记录
    master_program_to_delete = (
        master_program_df.alias("scpr")
        .join(
            scon_ids_to_delete.alias("del_ids"),
            (F.col("scpr.scpr_scon_id") == F.col("del_ids.scon_id")) &
            (F.col("scpr.scpr_mrkt_code") == F.col("del_ids.scon_mrkt_code")),
            "inner"
        )
        .select(F.col("scpr.*")).distinct()
    )
    
    # 4. 执行删除操作
    master_program_delete_count = 0
    if not master_program_to_delete.isEmpty():
        master_program_delta_table = DeltaTable.forName(spark, master_program_table_name)
        master_program_merge_result = (
            master_program_delta_table.alias("target")
            .merge(
                master_program_to_delete.alias("source"),
                """
                target.scpr_id = source.scpr_id AND
                target.scpr_mrkt_code = source.scpr_mrkt_code
                """
            )
            .whenMatchedDelete()
            .execute()
        )
        master_program_delete_count = max(master_program_to_delete.count(), 0)
    
    # 5. 插入数据到目标表
    if master_program_insert_count > 0:
        append_table(master_program_to_insert, master_program_table_name)
    print(f"master program inserted count: {master_program_insert_count}, deleted count: {master_program_delete_count}")
    
    # 6. KOR Region特殊处理：处理DELETE或失效的program记录，清空membershipnum并更新task_id
    # 对应SQL: update sconsumerprogram set scpr_membershipnum = '', task_id = task_id, scpr_end_dt = SRCC_SOURCETIMESTAMP
    # 查找tprogrammappingtodtl映射表中的有效program类型
    program_mapping_df = spark.table(f"{get_env_config('config_database')}.t_membership_program_code")
    valid_prgt_codes = program_mapping_df.select("prgt_code", "MarketCode").distinct()
    
    # 查找需要更新的program和consumer记录
    programs_to_update = (
        master_program_df.alias("scpr")
        .join(
            master_consumer_df.alias("scon"),
            (F.col("scpr.scpr_mrkt_code") == F.col("scon.scon_mrkt_code")) &
            (F.col("scpr.scpr_scon_id") == F.col("scon.scon_id")),
            "inner"
        )
        .join(
            spark.table(f"{get_env_config('silver_consumer_cleansed_database')}.t_clean_consumer")
                .where(f"TASK_ID = '{task_id}'")
                .filter(F.col("IS_INCLUDE") == True)
                .alias("srcc"),
            (F.col("scon.scon_srcs_code") == F.col("srcc.srcc_srcs_code")) &
            (F.col("scon.scon_mrkt_code") == F.col("srcc.srcc_mrkt_code")) &
            (F.col("scon.scon_brnd_code") == F.col("srcc.srcc_brnd_code")) &
            (F.col("scon.scon_consumerid") == F.col("srcc.srcc_consumerid")),
            "inner"
        )
        .join(
            itermediate_program_df.alias("srpg"),
            (F.col("srpg.srpg_mrkt_code") == F.col("scpr.scpr_mrkt_code")) &
            (F.col("srpg.SRPG_PRGT_CODE") == F.col("scpr.scpr_prgt_code")),
            "left"
        )
        .join(
            valid_prgt_codes.alias("mapping"),
            (F.col("scpr.scpr_prgt_code") == F.col("mapping.prgt_code")) &
            (F.col("scpr.scpr_mrkt_code") == F.col("mapping.MarketCode")),
            "inner"
        )
        .where(
            # 只有KOR才有这个逻辑
            (F.col("srcc.srcc_mrkt_code") == "KOR") &
            (F.col("srcc.SRCC_SOURCETIMESTAMP") >= F.col("scon.scon_sourcetimestamp")) &
            (F.coalesce(F.col("scpr.scpr_membershipnum"), F.lit("")) != "") &
            ((F.upper(F.col("srcc.srcc_action")) == "DELETE") | (F.col("srpg.SRPG_PRGT_CODE").isNull()))
        )
        .select(
            F.col("scpr.scpr_mrkt_code"),
            F.col("scpr.scpr_id"),
            F.col("scon.scon_id"),
            F.col("srcc.SRCC_SOURCETIMESTAMP")
        )
        .distinct()
    )
    
    # 更新program表：清空membershipnum，设置end_dt
    if not programs_to_update.isEmpty():
        # 更新program表
        master_program_delta_table = DeltaTable.forName(spark, master_program_table_name)
        master_program_delta_table.alias("target").merge(
            programs_to_update.alias("source"),
            """
            target.scpr_id = source.scpr_id AND
            target.scpr_mrkt_code = source.scpr_mrkt_code
            """
        ).whenMatchedUpdate(set={
            "scpr_membershipnum": F.lit(""),
            "scpr_end_dt": F.col("source.SRCC_SOURCETIMESTAMP"),
            "scpr_update_dt": F.current_timestamp(),
            "scpr_update_uid": F.lit("ELC")
        }).execute()
        
        # 更新consumer表task_id
        master_consumer_delta_table = DeltaTable.forName(spark, master_consumer_table_name)
        consumer_ids_to_update = programs_to_update.select("scon_id", "scpr_mrkt_code").distinct()
        master_consumer_delta_table.alias("target").merge(
            consumer_ids_to_update.alias("source"),
            """
            target.scon_id = source.scon_id AND
            target.scon_mrkt_code = source.scpr_mrkt_code
            """
        ).whenMatchedUpdate(set={
            "task_id": F.lit(task_id),
            "scon_update_dt": F.current_timestamp(),
            "scon_update_uid": F.lit("ELC")
        }).execute()
        
        update_count = programs_to_update.count()
        print(f"Updated {update_count} program records (cleared membershipnum and update task_id)")

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")

with StepLogger("5.3_generate_master_survive_tables", "05-3", "consumerlist", task_id=task_id) as logger:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")

    calc_consumer_phone_master(task_id)
    calc_consumer_address_master(task_id)
    calc_consumer_optin_master(task_id)
    calc_consumer_program_master(task_id)